# Week 2 Project 2 — Airline Tool-Calling Assistant with Ollama and SQLite

This notebook adapts the Week 2 lessons on prompt-based tool calling, multiple tool calls, Gradio, and SQLite integration. It uses a small local flight database and a local Ollama model. The assistant never needs OpenAI, Claude, Gemini, DALL-E, or a cloud API key.

## What this project demonstrates

- A model proposes a structured tool action as JSON.
- Python validates the action and executes the real function.
- SQLite supplies flight and booking state.
- The tool result is returned to Ollama for a final natural-language answer.
- A Gradio chat UI provides the user-facing layer.
- A small SVG flight card demonstrates a local visual response without a paid image API.

In [ ]:
import json
import re
import sqlite3
from datetime import datetime, timezone
from html import escape
from typing import Any

import requests
from IPython.display import SVG, Markdown, display

OLLAMA_URL = "http://127.0.0.1:11434"
MODEL = "llama3.2:1b"
REQUEST_TIMEOUT = 180

In [ ]:
db = sqlite3.connect(":memory:")
db.row_factory = sqlite3.Row
db.executescript("""
CREATE TABLE flights (
    id TEXT PRIMARY KEY,
    origin TEXT NOT NULL,
    destination TEXT NOT NULL,
    depart TEXT NOT NULL,
    arrive TEXT NOT NULL,
    cabin TEXT NOT NULL,
    seats INTEGER NOT NULL,
    price_usd INTEGER NOT NULL
);
CREATE TABLE bookings (
    code TEXT PRIMARY KEY,
    flight_id TEXT NOT NULL,
    passenger TEXT NOT NULL,
    created_at TEXT NOT NULL,
    FOREIGN KEY(flight_id) REFERENCES flights(id)
);
""")
db.executemany("INSERT INTO flights VALUES (?, ?, ?, ?, ?, ?, ?, ?)", [
    ("EG101", "Cairo", "London", "2026-10-10 09:00", "2026-10-10 14:00", "economy", 4, 420),
    ("EG102", "Cairo", "London", "2026-10-10 18:30", "2026-10-10 23:30", "business", 2, 980),
    ("EG201", "Cairo", "Dubai", "2026-10-11 07:15", "2026-10-11 12:30", "economy", 6, 310),
    ("EG301", "London", "Cairo", "2026-10-15 16:00", "2026-10-16 00:20", "economy", 3, 395),
])
db.commit()
print("Seeded", db.execute("SELECT COUNT(*) FROM flights").fetchone()[0], "flights")

In [ ]:
def search_flights(origin: str, destination: str, cabin: str = "economy") -> list[dict[str, Any]]:
    rows = db.execute(
        """SELECT * FROM flights
           WHERE lower(origin)=lower(?) AND lower(destination)=lower(?)
             AND lower(cabin)=lower(?) AND seats > 0
           ORDER BY depart""",
        (origin, destination, cabin),
    ).fetchall()
    return [dict(row) for row in rows]


def book_flight(flight_id: str, passenger: str) -> dict[str, Any]:
    flight = db.execute("SELECT * FROM flights WHERE id=? AND seats>0", (flight_id,)).fetchone()
    if flight is None:
        return {"ok": False, "error": "Flight not found or sold out."}
    code = f"W2{flight_id[-2:]}{db.execute('SELECT COUNT(*) FROM bookings').fetchone()[0] + 1:03d}"
    db.execute("INSERT INTO bookings VALUES (?, ?, ?, ?)", (code, flight_id, passenger, datetime.now(timezone.utc).isoformat()))
    db.execute("UPDATE flights SET seats=seats-1 WHERE id=?", (flight_id,))
    db.commit()
    return {"ok": True, "booking_code": code, "flight": dict(flight), "passenger": passenger}


def lookup_booking(code: str) -> dict[str, Any]:
    row = db.execute(
        """SELECT b.code, b.passenger, b.created_at, f.*
           FROM bookings b JOIN flights f ON f.id=b.flight_id WHERE b.code=?""", (code.upper(),)
    ).fetchone()
    return dict(row) if row else {"ok": False, "error": "Booking not found."}

TOOLS = {
    "search_flights": search_flights,
    "book_flight": book_flight,
    "lookup_booking": lookup_booking,
}

In [ ]:
def ollama_json(prompt: str) -> dict[str, Any] | None:
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": "Return only valid JSON. Never add markdown fences."},
            {"role": "user", "content": prompt},
        ],
        "stream": False,
        "format": "json",
        "options": {"temperature": 0, "num_ctx": 4096},
    }
    response = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    text = response.json()["message"]["content"]
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.S)
        return json.loads(match.group(0)) if match else None


def heuristic_action(text: str) -> dict[str, Any]:
    """Small deterministic fallback so the notebook remains useful on a 1B model."""
    lower = text.lower()
    if "booking" in lower and (code := re.search(r"\bW2[A-Z0-9]{4,}\b", text, re.I)):
        return {"tool": "lookup_booking", "arguments": {"code": code.group(0).upper()}}
    city_pairs = [("cairo", "london"), ("cairo", "dubai"), ("london", "cairo")]
    for origin, destination in city_pairs:
        if origin in lower and destination in lower:
            cabin = "business" if "business" in lower else "economy"
            return {"tool": "search_flights", "arguments": {"origin": origin.title(), "destination": destination.title(), "cabin": cabin}}
    return {"tool": "none", "arguments": {}}


def choose_action(text: str) -> dict[str, Any]:
    prompt = f"""Choose exactly one action for this airline request.
Available actions:
- search_flights arguments: origin, destination, cabin (economy or business)
- lookup_booking arguments: code
- none arguments: {{}}
Request: {text}
Return JSON with keys tool and arguments."""
    try:
        action = ollama_json(prompt)
        if isinstance(action, dict) and action.get("tool") in {*TOOLS, "none"}:
            return action
    except (requests.RequestException, ValueError, TypeError) as exc:
        print("Model JSON unavailable; using deterministic fallback:", exc)
    return heuristic_action(text)

In [ ]:
def run_airline_agent(user_text: str) -> dict[str, Any]:
    action = choose_action(user_text)
    tool_result: Any = {"ok": True, "message": "No database action was needed."}
    if action.get("tool") in TOOLS:
        try:
            tool_result = TOOLS[action["tool"]](**action.get("arguments", {}))
        except (TypeError, sqlite3.Error) as exc:
            tool_result = {"ok": False, "error": str(exc)}
    final_prompt = f"""You are an airline support assistant. Answer using only the tool result below.
Be concise. Never invent flight numbers, prices, seats, or booking codes.
User request: {user_text}
Chosen action: {json.dumps(action)}
Tool result: {json.dumps(tool_result, default=str)}
"""
    try:
        answer = ollama_json(final_prompt) if False else None
        response = requests.post(f"{OLLAMA_URL}/api/chat", json={
            "model": MODEL,
            "messages": [{"role": "system", "content": "You are a precise airline assistant."}, {"role": "user", "content": final_prompt}],
            "stream": False,
            "options": {"temperature": 0.2, "num_ctx": 4096},
        }, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        answer = response.json()["message"]["content"].strip()
    except requests.RequestException as exc:
        answer = f"Tool result (Ollama unavailable): {json.dumps(tool_result, default=str)}"
    return {"answer": answer, "action": action, "tool_result": tool_result}

In [ ]:
if True:
    demo_result = run_airline_agent("Find an economy flight from Cairo to London")
    display(Markdown(demo_result["answer"]))
    print("Action:", demo_result["action"])
    print("Tool result:", demo_result["tool_result"])

In [ ]:
def flight_card_svg(result: dict[str, Any]) -> str:
    data = result.get("tool_result", {})
    flight = data.get("flight") if isinstance(data, dict) else None
    if not flight and isinstance(data, list) and data:
        flight = data[0]
    if not flight and isinstance(data, dict) and data.get("id"):
        flight = data
    if not flight:
        return "<svg xmlns='http://www.w3.org/2000/svg' width='640' height='160'><text x='20' y='50' font-size='20'>No flight card available</text></svg>"
    title = escape(f"{flight['origin']} → {flight['destination']}")
    details = escape(f"{flight['id']}  |  {flight['depart']}  |  ${flight['price_usd']}  |  {flight['cabin']}")
    return f"""<svg xmlns='http://www.w3.org/2000/svg' width='640' height='180' viewBox='0 0 640 180'>
      <rect width='640' height='180' rx='18' fill='#102a43'/>
      <text x='28' y='58' fill='white' font-size='28' font-family='sans-serif'>{title}</text>
      <text x='28' y='100' fill='#d9e2ec' font-size='17' font-family='sans-serif'>{details}</text>
      <text x='28' y='145' fill='#7cc4fa' font-size='15' font-family='sans-serif'>Local Ollama + SQLite result</text>
    </svg>"""

if "tool_result" in demo_result:
    display(SVG(flight_card_svg(demo_result)))

## Gradio interface

This UI uses the same agent function. It does not expose the database directly; every request goes through the action selection and tool execution path.

In [ ]:
try:
    import gradio as gr
except ImportError:
    gr = None
    print("Install the UI dependency with: python -m pip install 'gradio>=5,<6'")


def ui_reply(message: str, history: list[dict[str, str]]) -> str:
    result = run_airline_agent(message)
    return result["answer"]

if gr is not None:
    agent_demo = gr.ChatInterface(
        fn=ui_reply,
        title="Week 2 — Airline Tool-Calling Assistant",
        description="Try: Find an economy flight from Cairo to London. The database is local and in memory.",
    )
    RUN_UI = False
    if RUN_UI:
        agent_demo.launch(inbrowser=False, share=False)
    else:
        print("UI built. Set RUN_UI = True and rerun this cell to launch it.")

## Exercises

1. Add a `cancel_booking` tool and a confirmation step before mutating SQLite.
2. Add a second tool call that searches flights and then formats the best result.
3. Replace the seeded data with a local CSV import.
4. Add a microphone or text-to-speech layer as an optional front end. Keep the Ollama text agent and tools unchanged.

The original course's cloud multimodal examples use provider APIs. This local adaptation intentionally keeps the core agent and tool-calling behavior offline; the SVG card is a deterministic local visual artifact.